## MMTHE01 - Masters Thesis

### B. Thesis - Imputing missing fields and encode data for remaining positive data and get final positive
* Data Encoding
* Feature Scaling
* Get final dataset with the final positive class features selected in Step D

#### Importing the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import pickle

from sklearn.preprocessing import LabelEncoder

In [2]:
# Check your current working directory
cwd = os.getcwd()

# Define your relative path
relative_path = r"Analysis"  # adjust this relative to cwd

# Build the full path
full_path = os.path.join(cwd, relative_path)

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

Changed directory to: C:\Users\eaber\Documents\Masters Thesis Final\Analysis


#### Importing the dataset

In [3]:
### Mark categorical fields to ensure they are appropriately delt with
categorical_columns_all = ['term', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'issue_d', 'purpose', 'title',
                       'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status']

In [4]:
positive_dataset = pd.read_csv('remaining_positive_lending_club.csv', dtype={col: 'category' for col in categorical_columns_all}, low_memory=False)

In [5]:
### Import final features from previous step
intermediate_output_path = os.path.join(r'intermediate_output', 'final_features_to_stay.csv')
features_to_stay =  pd.read_csv(intermediate_output_path)

In [6]:
features_to_stay_list = features_to_stay["features_to_stay"].tolist()
len(features_to_stay_list)

62

### Dataset after Step 1

In [7]:
dataset_final = positive_dataset[features_to_stay_list].copy()

In [8]:
dataset_final.head()

,loan_amnt,term,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,purpose,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,year_issued,default_status
0,8000,36 months,A1,Meat Cutter,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2018,credit_card,...,98.0,25.0,0.0,0.0,232617.0,20910.0,16700.0,23181.0,2018,1
1,6000,36 months,C2,NaN,NaN,MORTGAGE,17000.0,Not Verified,Dec-2018,debt_consolidation,...,80.0,0.0,0.0,0.0,13200.0,5005.0,4100.0,5400.0,2018,1
2,10000,36 months,A5,NaN,NaN,MORTGAGE,56000.0,Verified,Nov-2018,debt_consolidation,...,100.0,50.0,1.0,0.0,327790.0,30771.0,12000.0,34590.0,2018,1
3,10000,60 months,E5,CNC Robotic Welder,10+ years,RENT,70000.0,Source Verified,Nov-2018,debt_consolidation,...,91.7,100.0,0.0,0.0,25493.0,23289.0,1800.0,21693.0,2018,1
4,35000,36 months,C4,NaN,< 1 year,MORTGAGE,300000.0,Source Verified,Nov-2018,other,...,95.5,44.4,1.0,0.0,142518.0,87111.0,20200.0,90118.0,2018,1


In [9]:
dataset_final.head(100).to_csv('intermediate_output/positive_dataset_final.csv', index=False)

In [10]:
def top_missing_cols_perc(df):
    """
    returns missing columns in dataframe with missing values percent > thresh
    if n=None. It will gave whole dataframe with missing values percent > thresh
    """
    
    dff = (df.isnull().sum()/df.shape[0])*100
    dff = dff.reset_index()
    dff.columns = ['col','missing_percent']
    dff = dff.sort_values(by=['missing_percent'],ascending=True).reset_index(drop=True)
    return dff

In [11]:
missing_perc_matrix = top_missing_cols_perc(dataset_final)

In [12]:
missing_perc_matrix.to_csv('intermediate_output/missing_perc_matrix_values3.csv', index=False)

### Step 2: Use SimpleImputer to impute missing feature values

#### 2.1 Impute missing values for categorical variables

##### 2.1.1 Get the cat features

In [13]:
columns_with_missing = dataset_final.columns[dataset_final.isnull().any()].to_list()
len(columns_with_missing)

40

In [14]:
dict_ = {}
for column in columns_with_missing:
    dict_[column] = dataset_final.dtypes[column]

In [15]:
#### Get caregorical feature columns with missing data plus the columns without missing data
categorical_columns = [column for column in columns_with_missing if dataset_final[column].dtype == 'category']
len(categorical_columns)

3

In [16]:
categorical_columns_dataset = dataset_final[categorical_columns].copy()
categorical_columns_dataset.head()

,emp_title,emp_length,title
0,Meat Cutter,10+ years,Credit card refinancing
1,NaN,NaN,Debt consolidation
2,NaN,NaN,Debt consolidation
3,CNC Robotic Welder,10+ years,Debt consolidation
4,NaN,< 1 year,Other


In [17]:
cat_missing_check = categorical_columns_dataset.columns[categorical_columns_dataset.isnull().any()].to_list()
len(cat_missing_check)

3

##### 2.1.2 Impute missing categorical values

In [18]:
from sklearn.impute import SimpleImputer

In [19]:
cat_imputer = SimpleImputer(strategy='most_frequent')

In [20]:
imputed_cat_data = cat_imputer.fit_transform(categorical_columns_dataset)

In [21]:
imputed_cat_df = pd.DataFrame(imputed_cat_data, columns=categorical_columns_dataset.columns)

In [22]:
imputed_cat_df.head()

,emp_title,emp_length,title
0,Meat Cutter,10+ years,Credit card refinancing
1,Manager,10+ years,Debt consolidation
2,Manager,10+ years,Debt consolidation
3,CNC Robotic Welder,10+ years,Debt consolidation
4,Manager,< 1 year,Other


In [23]:
cat_missing_check = imputed_cat_df.columns[imputed_cat_df.isnull().any()].to_list()
cat_missing_check

[]

#### 2.2 Impute missing values for numeric variable

##### 2.2.1 Get the numeric features

In [24]:
columns_with_missing = dataset_final.columns[dataset_final.isnull().any()].to_list()
len(columns_with_missing)

40

In [25]:
dict_ = {}
for column in columns_with_missing:
    dict_[column] = dataset_final.dtypes[column]

In [26]:
#### Get numeric feature columns with missing data plus the columns without missing data
numeric_columns = [column for column in columns_with_missing if dataset_final[column].dtype == 'float64']
len(numeric_columns)

37

In [27]:
numeric_columns_dataset = dataset_final[numeric_columns].copy()
numeric_columns_dataset.head()

,dti,revol_util,collections_12_mths_ex_med,tot_coll_amt,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,17.83,29.1,0.0,0.0,37800.0,12.0,15848.0,7592.0,54.5,0.0,...,0.0,6.0,98.0,25.0,0.0,0.0,232617.0,20910.0,16700.0,23181.0
1,19.99,8.3,1.0,624.0,7800.0,3.0,626.0,3759.0,8.3,0.0,...,1.0,2.0,80.0,0.0,0.0,0.0,13200.0,5005.0,4100.0,5400.0
2,20.53,53.3,0.0,0.0,19700.0,5.0,43118.0,2323.0,80.6,0.0,...,0.0,2.0,100.0,50.0,1.0,0.0,327790.0,30771.0,12000.0,34590.0
3,11.37,84.0,0.0,0.0,3800.0,7.0,5822.0,232.0,87.1,0.0,...,0.0,3.0,91.7,100.0,0.0,0.0,25493.0,23289.0,1800.0,21693.0
4,13.88,84.5,0.0,0.0,52400.0,4.0,3960.0,5173.0,74.4,0.0,...,0.0,2.0,95.5,44.4,1.0,0.0,142518.0,87111.0,20200.0,90118.0


In [28]:
num_missing_check = numeric_columns_dataset.columns[numeric_columns_dataset.isnull().any()].to_list()
len(num_missing_check)

37

##### 2.2.2 Impute missing numeric values

In [29]:
num_imputer = SimpleImputer(strategy='mean')

In [30]:
imputed_num_data = num_imputer.fit_transform(numeric_columns_dataset)

In [31]:
imputed_num_df = pd.DataFrame(imputed_num_data, columns=numeric_columns_dataset.columns)

In [32]:
imputed_num_df.head()

,dti,revol_util,collections_12_mths_ex_med,tot_coll_amt,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,17.83,29.1,0.0,0.0,37800.0,12.0,15848.0,7592.0,54.5,0.0,...,0.0,6.0,98.0,25.0,0.0,0.0,232617.0,20910.0,16700.0,23181.0
1,19.99,8.3,1.0,624.0,7800.0,3.0,626.0,3759.0,8.3,0.0,...,1.0,2.0,80.0,0.0,0.0,0.0,13200.0,5005.0,4100.0,5400.0
2,20.53,53.3,0.0,0.0,19700.0,5.0,43118.0,2323.0,80.6,0.0,...,0.0,2.0,100.0,50.0,1.0,0.0,327790.0,30771.0,12000.0,34590.0
3,11.37,84.0,0.0,0.0,3800.0,7.0,5822.0,232.0,87.1,0.0,...,0.0,3.0,91.7,100.0,0.0,0.0,25493.0,23289.0,1800.0,21693.0
4,13.88,84.5,0.0,0.0,52400.0,4.0,3960.0,5173.0,74.4,0.0,...,0.0,2.0,95.5,44.4,1.0,0.0,142518.0,87111.0,20200.0,90118.0


In [33]:
num_missing_check = imputed_num_df.columns[imputed_num_df.isnull().any()].to_list()
num_missing_check

[]

### Step 3: Combining the data, feature engineering and encoding categorical data using the label encoder

#### 3.1 Combine the data

In [34]:
full_columns = dataset_final.columns[dataset_final.notna().all()].to_list()

In [35]:
dataset_full_features = dataset_final[full_columns]

In [36]:
dataset_full_features.head(1)

,loan_amnt,term,sub_grade,home_ownership,annual_inc,verification_status,issue_d,purpose,zip_code,addr_state,...,inq_last_6mths,open_acc,pub_rec,revol_bal,total_acc,initial_list_status,policy_code,delinq_amnt,year_issued,default_status
0,8000,36 months,A1,MORTGAGE,55000.0,Not Verified,Dec-2018,credit_card,984xx,WA,...,0.0,12.0,0.0,10999,49.0,w,1,0.0,2018,1


In [37]:
dataset_j1 = dataset_full_features.merge(imputed_cat_df, on= imputed_cat_df.index ,how='inner' )
dataset_j1.drop('key_0', axis=1, inplace=True)

In [38]:
dataset_j1.head(1)

,loan_amnt,term,sub_grade,home_ownership,annual_inc,verification_status,issue_d,purpose,zip_code,addr_state,...,revol_bal,total_acc,initial_list_status,policy_code,delinq_amnt,year_issued,default_status,emp_title,emp_length,title
0,8000,36 months,A1,MORTGAGE,55000.0,Not Verified,Dec-2018,credit_card,984xx,WA,...,10999,49.0,w,1,0.0,2018,1,Meat Cutter,10+ years,Credit card refinancing


In [39]:
full_dataset = dataset_j1.merge(imputed_num_df, on= imputed_num_df.index ,how='inner' )
full_dataset.drop('key_0', axis=1, inplace=True)

In [40]:
full_dataset.head(1)

,loan_amnt,term,sub_grade,home_ownership,annual_inc,verification_status,issue_d,purpose,zip_code,addr_state,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,8000,36 months,A1,MORTGAGE,55000.0,Not Verified,Dec-2018,credit_card,984xx,WA,...,0.0,6.0,98.0,25.0,0.0,0.0,232617.0,20910.0,16700.0,23181.0


In [41]:
full_dataset.shape

(235517, 62)

In [42]:
full_data_missing_check = full_dataset.columns[full_dataset.isnull().any()].to_list()
full_data_missing_check

[]

#### 3.2: Dealing with Date Features

In [43]:
full_dataset[['issue_d','earliest_cr_line']].head()

,issue_d,earliest_cr_line
0,Dec-2018,May-2002
1,Dec-2018,Nov-2000
2,Nov-2018,Apr-2011
3,Nov-2018,Nov-2008
4,Nov-2018,Aug-2010


In [44]:
full_dataset['issue_year'] = full_dataset['issue_d'].apply(lambda x: x.split('-')[1])
full_dataset['issue_month'] = full_dataset['issue_d'].apply(lambda x: x.split('-')[0])

In [45]:
full_dataset['earliest_cr_line_year'] = full_dataset['earliest_cr_line'].apply(lambda x: str(x).split('-')[1])
full_dataset['earliest_cr_line_month'] = full_dataset['earliest_cr_line'].apply(lambda x: str(x).split('-')[0])

In [46]:
full_dataset['term_in_months'] = full_dataset['term'].apply(lambda x: x.split(' ')[1])

In [47]:
full_dataset[['issue_d','earliest_cr_line','issue_year','issue_month','earliest_cr_line_year','earliest_cr_line_month','term_in_months']].head()

,issue_d,earliest_cr_line,issue_year,issue_month,earliest_cr_line_year,earliest_cr_line_month,term_in_months
0,Dec-2018,May-2002,2018,Dec,2002,May,36
1,Dec-2018,Nov-2000,2018,Dec,2000,Nov,36
2,Nov-2018,Apr-2011,2018,Nov,2011,Apr,36
3,Nov-2018,Nov-2008,2018,Nov,2008,Nov,60
4,Nov-2018,Aug-2010,2018,Nov,2010,Aug,36


In [48]:
### Drop the raw date features
full_dataset = full_dataset.drop(['issue_d','earliest_cr_line','term'],axis=1)

#### 3.4 Export data Unencoded data

In [49]:
# Save all datasets in one file
with open("saved_data/final_dataset_unencoded.pkl", "wb") as f:
    pickle.dump((full_dataset), f)

In [50]:
full_dataset.shape

(235517, 64)

#### 3.3 Encode categorical data

##### 3.2.1 Understanding the categorical variables which need encoding

In [51]:
#list of categorical features which require encoding 
categorical_cols = ['sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'title', 'issue_month', 
                    'earliest_cr_line_month', 'zip_code', 'addr_state', 'initial_list_status']

In [52]:
# Count unique values in each categorical column
category_counts = full_dataset[categorical_cols].nunique()

print(category_counts)

sub_grade                    35
emp_title                 86332
emp_length                   11
home_ownership                6
verification_status           3
purpose                      14
title                     11035
issue_month                  12
earliest_cr_line_month       12
zip_code                    897
addr_state                   50
initial_list_status           2
dtype: int64


In [53]:
### Encoding methods to use:
#1. Use Label encoding - sub_grade, emp_title, title, zip_code, addr_state (because of high cardinality)
#2. Using One Hot Encoding (OHE) - emp_length, home_ownership, verification_status, purpose, issue_month, earliest_cr_line_month, initial_list_status (low cardinality so it is efficient to use OHE)

##### 3.2.2 Label Encoding

In [54]:
def label_encode_keep_nan(series):
    """
    Label encodes a pandas Series while leaving NaNs untouched.
    Returns the encoded series and the fitted encoder.
    """
    # Keep mask of missing values
    mask = series.isna()

    # Apply LabelEncoder to non-NaN values only
    le = LabelEncoder()
    encoded = le.fit_transform(series[~mask])

    # Create a full-length array with NaNs
    full_encoded = pd.Series(np.nan, index=series.index)
    full_encoded[~mask] = encoded

    return full_encoded, le

In [55]:
full_dataset['sub_grade'], sub_grade_encoder = label_encode_keep_nan(full_dataset['sub_grade'])
full_dataset['emp_title'], emp_title_encoder = label_encode_keep_nan(full_dataset['emp_title'])
full_dataset['title'], title_encoder = label_encode_keep_nan(full_dataset['title'])
full_dataset['zip_code'], zip_code_encoder = label_encode_keep_nan(full_dataset['zip_code'])
full_dataset['addr_state'], addr_state_encoder = label_encode_keep_nan(full_dataset['addr_state'])

##### 3.2.3 One Hot Encoding

In [56]:
from sklearn.preprocessing import OneHotEncoder

In [57]:
cols_for_ohe = ['emp_length', 'home_ownership', 'verification_status', 'purpose', 'issue_month', 'earliest_cr_line_month', 'initial_list_status']

In [58]:
# Initialize OHE encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [59]:
# Fit and transform
encoded_array = encoder.fit_transform(full_dataset[cols_for_ohe])

In [60]:
# Create new DataFrame with encoded column names
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(cols_for_ohe))

In [61]:
# Combine with the rest of the data
full_dataset_final = pd.concat([full_dataset.drop(columns=cols_for_ohe), encoded_df], axis=1)

In [62]:
full_dataset_final.head()

,loan_amnt,sub_grade,annual_inc,zip_code,addr_state,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,...,earliest_cr_line_month_Jan,earliest_cr_line_month_Jul,earliest_cr_line_month_Jun,earliest_cr_line_month_Mar,earliest_cr_line_month_May,earliest_cr_line_month_Nov,earliest_cr_line_month_Oct,earliest_cr_line_month_Sep,initial_list_status_f,initial_list_status_w
0,8000,0.0,55000.0,882.0,46.0,0.0,0.0,12.0,0.0,10999,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,6000,11.0,17000.0,322.0,9.0,1.0,0.0,8.0,0.0,650,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,10000,4.0,56000.0,772.0,3.0,0.0,1.0,7.0,1.0,10494,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,10000,24.0,70000.0,555.0,13.0,0.0,1.0,4.0,0.0,3193,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,35000,13.0,300000.0,886.0,46.0,0.0,2.0,22.0,1.0,44283,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [63]:
full_dataset_final.groupby('default_status').count()

,loan_amnt,sub_grade,annual_inc,zip_code,addr_state,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,...,earliest_cr_line_month_Jan,earliest_cr_line_month_Jul,earliest_cr_line_month_Jun,earliest_cr_line_month_Mar,earliest_cr_line_month_May,earliest_cr_line_month_Nov,earliest_cr_line_month_Oct,earliest_cr_line_month_Sep,initial_list_status_f,initial_list_status_w
default_status,,,,,,,,,,,,,,,,,,,,,
1,235517,235517,235517,235517,235517,235517,235517,235517,235517,235517,...,235517,235517,235517,235517,235517,235517,235517,235517,235517,235517


In [64]:
full_dataset_final['default_status'].value_counts(normalize=True) * 100

default_status
1    100.0
Name: proportion, dtype: float64

#### 3.4 Get final dataset with only the selected features

In [65]:
### Import the pickle saved dataset
with open("saved_data/selected_features_final.pkl","rb") as f:
    selected_features_final = pickle.load(f)

In [66]:
positive_dataset_final = full_dataset_final[selected_features_final]

In [67]:
positive_dataset_final.shape

(235517, 110)

#### 3.4 Export final positive dataset

In [68]:
# Save datasets in one file
with open("saved_data/positive_final_dataset_encoded.pkl", "wb") as f:
    pickle.dump((positive_dataset_final), f)